# Clustering

이번 실습에서는 이러한 것들을 다뤄보려 합니다:
- k-means 실습
- DBSCAN 실습

주로 사용할 라이브러리들의 정보는 다음과 같습니다:
- sklearn: 머신러닝 라이브러리로, k-means와 DBSCAN의 모델 학습 및 평가에 사용됩니다.
- matplotlib: 데이터 시각화에 사용되는 라이브러리입니다.
- pandas: 데이터프레임과 같은 자료구조를 통해 구조화된 데이터를 효율적으로 처리하고 분석할 수 있습니다.

In [1]:
# 필요한 라이브러리들 import
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, normalized_mutual_info_score, adjusted_rand_score

# k-means

k-means는 널리 사용되는 클러스터링 알고리즘 중 하나로, 비슷한 특성을 가진 데이터들을 그룹으로 묶는 데 사용됩니다. k-means는 이해하기 쉽고 구현이 간단하기 때문에 실제로 많이 사용되며, 다양한 변형 알고리즘도 많이 존재합니다.

k-means를 사용해서 주어진 데이터셋을 클러스터링 해봅시다.

주어진 데이터셋을 먼저 살펴보겠습니다.
pandas 라이브러리의 `read_csv()` 함수를 사용해서 데이터를 불러와줍시다.

NOTE: 데이터는 제공된 구글드라이브에서 확인하실 수 있습니다. 데이터를 다운로드 하신 후에, 코랩에 업로드 해서 사용해주세요.

## 1. 데이터 확인 및 전처리

In [ ]:
# 주어진 데이터셋 불러오기(artset1.csv)
# df = pd.read_csv('./artset1.csv')
df = pd.read_csv('./artset1.csv')

df.head()

HTTPError: HTTP Error 404: Not Found

데이터의 첫번째 값은 포인트 이름, 마지막 값은 true label입니다.

모델을 학습시킬 때는 해당 정보들을 사용하지 않을 것이기 때문에, 데이터를 전처리 하며 포인트 이름은 지우고, true label은 따로 보관하도록 하겠습니다.

In [ ]:
# point, answer 컬럼을 제외한 나머지 부분 import
coordinates = df.drop(['point', 'answer'], axis=1)
target = df['answer']

다음으론, matplotlib 라이브러리를 사용해서 데이터를 시각화 해봅시다. 이전에 데이터에서 불러온 data의 coordinates 값을 2차원 공간에 plot 해보도록 하겠습니다.

In [ ]:
# matplotlib 라이브러리를 사용해서 이미지 plot
# 불러올 데이터 지정
plt.scatter(coordinates.iloc[:, 0], coordinates.iloc[:, 1])

# 그래프 제목 지정
plt.title('artset1')

# x축, y축 라벨 지정
plt.xlabel('x')
plt.ylabel('y')

# 그래프 출력
plt.show()

## 2. k-means 모델 학습

이제 데이터를 성공적으로 불러왔기 때문에, 데이터를 사용해서 모델을 학습시켜보도록 하겠습니다. scikit-learn 라이브러리에 내장되어있는 KMeans 모델을 사용하도록 하겠습니다.

모델을 새로 만들때, 우린 클러스터링 갯수를 지정해줘야 합니다. 위의 plot 이미지에서 클러스터의 갯수가 15개인 것을 확인할 수 있기 때문에 우린 15개로 n_cluster 파라미터를 지정해주겠습니다.

In [ ]:
kmeans = KMeans(n_clusters=15, random_state=0)
kmeans.fit(coordinates)

우리의 모델이 데이터를 학습했으니, 학습된 결과물을 확인해보도록 하겠습니다. 모델 결과물의 centroid와 predicted label을 시각화 해봅시다.

In [ ]:
# 모델의 centroid 정보 가져오기
centroids = kmeans.cluster_centers_

# 모델의 predicted label 가져오기
predicted_labels = kmeans.labels_

In [ ]:
plt.scatter(coordinates.iloc[:, 0], coordinates.iloc[:, 1], c=predicted_labels)

plt.scatter(centroids[:, 0], centroids[:, 1], c='red', marker='x')
plt.title('Clustering Result')
plt.xlabel('x')
plt.ylabel('y')
plt.show()

## 3. 모델 평가

모델을 평가해봅시다. 우리는 크게 두가지 metric을 사용할 것입니다.
1. silhouette score
2. NMI(Normalized Mutual Information)

가장 먼저 사용해 볼 metric은 silhoutte score입니다. 이는 각 데이터 포인트가 자신이 속한 클러스터와 얼마나 잘 맞는지, 그리고 다른 클러스터와 얼마나 구별되는지를 나타냅니다.

Silhouette Score는 -1에서 1 사이의 값을 가지며, 값이 클수록 더 나은 클러스터링을 의미합니다. 1에 가까울수록 해당 데이터 포인트가 적절한 클러스터에 속해 있으며, 0에 가까울수록 경계에 위치해 있다는 것을 의미하고, -1에 가까울수록 잘못된 클러스터에 속해 있다는 것을 나타냅니다.

In [ ]:
silhouette_avg = silhouette_score(coordinates, predicted_labels)
print(f'Silhouette Score: {silhouette_avg}')

NMI는 0에서 1 사이의 값을 가지며, 1에 가까울수록 두 클러스터링(predicted and target) 결과가 동일함을 의미합니다. NMI는 클러스터의 크기와 분포에 관계없이 두 클러스터링의 일치도를 평가할 수 있어, 클러스터링 알고리즘의 성능을 비교하는 데 유용합니다.

In [ ]:
nmi_score = normalized_mutual_info_score(target, predicted_labels)
print(f'NMI Score: {nmi_score}')

# 📝 실습 과제: k-means

**k-means을 사용해서, network.csv 데이터를 클러스터링 하고, 모델의 NMI score를 출력해주세요.**

In [ ]:
# 이 cell의 코드는 수정하지 않으셔도 됩니다!
df = pd.read_csv('./network.csv')

df.head()

In [ ]:
# 데이터 전처리 (target value 분리)




In [ ]:
# matplotlib으로 데이터 plot 하기




In [ ]:
# k-means 모델 학습시키기




In [ ]:
# 학습시킨 결과물 출력하기




In [ ]:
# 모델의 NMI score 출력하기




# DBSCAN

DBSCAN은 density-based 클러스터링 알고리즘으로, 데이터의 밀도에 따라 클러스터를 형성하고, 밀도가 낮은 지역의 데이터 포인트를 노이즈로 간주하는 알고리즘입니다. DBSCAN은 클러스터의 모양이나 크기에 민감하지 않아, 복잡한 형태의 클러스터를 잘 찾아낼 수 있습니다.

특히 k-means가 구 형태의 클러스터를 찾으려는 경향을 보이는데 비해 DBSCAN은 여러 형태의 클러스터를 찾아낼 수 있습니다.

## 1. k-means가 잘 동작하지 않는 경우

circles.csv 데이터를 가져와서 시각화 해보도록 하겠습니다.

In [ ]:
# 주어진 데이터셋 불러오기
df = pd.read_csv('./circles.csv')

coordinates = df.drop(['point', 'answer'], axis=1)
target = df['answer']

In [ ]:
# matplotlib 라이브러리를 사용해서 이미지 plot
plt.scatter(coordinates.iloc[:, 0], coordinates.iloc[:, 1], c=target, cmap='jet')
plt.title('circle')
plt.xlabel('x')
plt.ylabel('y')
plt.show()

잘 불러와진 것을 확인할 수 있습니다. 이제 k-means를 사용해서 클러스터링을 한 후에, 결과물이 어떤지 확인해보도록 하겠습니다.

In [ ]:
# k-means 학습
kmeans = KMeans(n_clusters=3, random_state=0)
kmeans.fit(coordinates)

centroids = kmeans.cluster_centers_
predicted_labels = kmeans.labels_

In [ ]:
# k-means 결과물 시각화
plt.scatter(coordinates.iloc[:, 0], coordinates.iloc[:, 1], c=predicted_labels)
plt.scatter(centroids[:, 0], centroids[:, 1], c='red', marker='x')

plt.title('Clustering Result')
plt.xlabel('x')
plt.ylabel('y')
plt.show()

In [ ]:
# NMI score 출력
nmi_score = normalized_mutual_info_score(target, predicted_labels)
print(f'NMI Score: {nmi_score}')

보시면 알겠지만, 우리가 원하던 대로 클러스터링이 진행되지도 않았을 뿐더러 NMI score도 처참합니다...

이러한 경우에 적용하기 좋은 알고리즘이 DBSCAN 입니다. DBSCAN은 데이터 사이의 밀도를 기반으로 클러스터하기 때문에, circle 데이터셋과 같이 복잡한 모양의 클러스터도 잘 잡아냅니다. DBSCAN 모델을 학습시키고 결과물을 확인해봅시다.

## 2. DBSCAN 모델 학습

k-means와 마찬가지로 scikit-learn에 구현되어있는 DBSCAN을 불러와서 사용하도록 하겠습니다.

In [ ]:
# DBSCAN 모델 학습
eps_value = 5 # Try: 5, 100

dbscan = DBSCAN(eps=eps_value, min_samples=4)
dbscan.fit(coordinates)

predicted_labels = dbscan.labels_

In [ ]:
plt.scatter(coordinates.iloc[:, 0], coordinates.iloc[:, 1], c=predicted_labels)
plt.title('Clustering Result')
plt.xlabel('x')
plt.ylabel('y')
plt.show()

### 📝 중간 실습: eps 값 찾아보기

결과물을 시각화해보니 클러스터링이 제대로 되지 않은 것을 확인할 수 있죠?

그 이유는 모델의 hyperparameter(eps, min_samples) 설정에 있습니다. k-means가 k에 크게 의존하듯이, DBSCAN 또한 eps과 min_samples에 크게 의존합니다. eps를 변경하면서 적절한 값을 찾아봅시다.

In [ ]:
# DBSCAN 모델 학습
eps_value =  # 적절한 eps 값을 찾아봅시다.

dbscan = DBSCAN(eps=eps_value, min_samples=4)
dbscan.fit(coordinates)

predicted_labels = dbscan.labels_

In [ ]:
plt.scatter(coordinates.iloc[:, 0], coordinates.iloc[:, 1], c=predicted_labels)
plt.title('Clustering Result')
plt.xlabel('x')
plt.ylabel('y')
plt.show()

## 3. 모델 평가

NMI score로 모델을 평가해보도록 하겠습니다.

NOTE: Silhouette score는 클러스터 내의 포인트 간 거리를 기반으로 점수를 매기기 때문에, 구 형태의 클러스터링에서 잘 작동합니다. circle 데이터셋과 같이 복잡한 형태의 클러스터들을 가지는 데이터셋을 평가하는데는 적합하지 않습니다.

In [ ]:
nmi_score = normalized_mutual_info_score(target, predicted_labels)
print(f'NMI Score: {nmi_score}')

k-means에 비해 훨씬 좋은 결과가 나온것을 확인할 수 있습니다!

# 📝 실습 과제: DBSCAN

**DBSCAN을 사용해서, face.csv 데이터를 클러스터링 하고, 모델의 NMI score를 출력해주세요.**

In [ ]:
# 이 cell의 코드는 수정하지 않으셔도 됩니다!
df = pd.read_csv('./face.csv')

df.head()

In [ ]:
# 데이터 전처리 (point 이름 지우기, target value 분리)




In [ ]:
# matplotlib으로 데이터 plot 하기




In [ ]:
# DBSCAN 모델 학습시키기




In [ ]:
# 학습시킨 결과물 출력하기




In [ ]:
# 모델의 NMI score 출력하기


